# 02 — Train the Intent Classifier via Mistral Classifier Factory

**Cookbook reference:** [Mistral Classifier Factory — Intent Classification](https://docs.mistral.ai/cookbooks/mistral-classifier_factory-intent_classification)

This notebook fine-tunes a `ministral-3b-latest` intent classifier on the Veridian
IT ticket data prepared in `01_data_prep.ipynb`, evaluates it on the held-out test
set, and saves the fine-tuned model ID for use in `03_agent_demo.ipynb`.

In [ ]:
%pip install -q mistralai pandas scikit-learn python-dotenv rich

In [ ]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

DATA_DIR        = Path("data")
TEST_FILE       = DATA_DIR / "test.jsonl"
MODEL_ID_FILE   = DATA_DIR / "classifier_model_id.txt"
CONFIG_FILE     = DATA_DIR / "finetune_config.json"

INTENT_LABELS = [
    "access_request",
    "security_incident",
    "hardware_issue",
    "software_issue",
    "onboarding",
    "payments_incident",
    "expense_request",
    "general_question",
]

## 1. Overview

### What we're training

A **single-label intent classifier** on 60 Veridian IT support tickets, fine-tuned
using Mistral's **Classifier Factory** — a supervised fine-tuning path that optimises
a model for discrete label prediction rather than next-token generation.

| Parameter | Value |
|---|---|
| Base model | `ministral-3b-latest` (recommended by cookbook for classification tasks) |
| Job type | `classifier` (Classifier Factory path, distinct from standard SFT) |
| Training steps | 100 |
| Learning rate | 4e-5 (0.00004) |
| Label key | `intent` |
| Classes | 8 (see `01_data_prep.ipynb`) |
| Typical training time | **5–15 minutes** on La Plateforme shared infrastructure |

### Why `ministral-3b-latest`?

The cookbook recommends a compact model for Classifier Factory jobs:
the classification head is what learns the label mapping, not the generative
capacity of the full model. A 3B model fine-tuned for classification is
faster and cheaper at inference than prompting a 70B+ model to guess intent,
and substantially more accurate on a specific label taxonomy.

This classifier replaces the routing work `mistral-large` would otherwise
have to perform during the agentic loop — freeing the large model to focus
entirely on reasoning and tool use.

## 2. Setup

File IDs are loaded from `data/finetune_config.json` written by `01_data_prep.ipynb`.
If you are running this notebook standalone, paste your file IDs into the
`MANUAL_TRAIN_FILE_ID` / `MANUAL_VAL_FILE_ID` variables below.

In [ ]:
# ── Manual override — paste values here if not running after Notebook 01 ──
MANUAL_TRAIN_FILE_ID = ""   # e.g. "file-abc123"
MANUAL_VAL_FILE_ID   = ""   # e.g. "file-def456"

# ── Load from config written by 01_data_prep.ipynb ────────────────────────
if CONFIG_FILE.exists() and not MANUAL_TRAIN_FILE_ID:
    cfg = json.loads(CONFIG_FILE.read_text())
    training_file_id   = cfg["train_file_id"]
    validation_file_id = cfg["val_file_id"]
    console.print(f"[green]Loaded file IDs from {CONFIG_FILE}[/green]")
elif MANUAL_TRAIN_FILE_ID:
    training_file_id   = MANUAL_TRAIN_FILE_ID
    validation_file_id = MANUAL_VAL_FILE_ID
    console.print("[yellow]Using manually pasted file IDs.[/yellow]")
else:
    raise FileNotFoundError(
        f"{CONFIG_FILE} not found and no manual IDs provided.\n"
        "Run 01_data_prep.ipynb first, or paste file IDs into the variables above."
    )

console.print(f"  training_file_id   = [bold]{training_file_id}[/bold]")
console.print(f"  validation_file_id = [bold]{validation_file_id}[/bold]")

In [ ]:
from mistralai import Mistral

# ── API key ───────────────────────────────────────────────────────────────
try:
    from google.colab import userdata  # type: ignore
    MISTRAL_API_KEY = userdata.get("MISTRAL_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

if not MISTRAL_API_KEY:
    raise EnvironmentError(
        "MISTRAL_API_KEY is not set.\n"
        "Colab: add it via Secrets (key icon in the left sidebar).\n"
        "Local: add MISTRAL_API_KEY=sk-... to a .env file."
    )

# To target a Forge endpoint instead of La Plateforme, add:
#   server_url="https://mistral.your-forge-deployment.internal/v1"
client = Mistral(api_key=MISTRAL_API_KEY)
console.print("[bold green]Mistral client initialised.[/bold green]")

## 3. Create fine-tuning job

The critical parameter is `job_type="classifier"` — this routes the job through
the Classifier Factory training path instead of standard supervised fine-tuning.
The training data format (`{"text": ..., "labels": {"intent": ...}}`) is also
specific to this path and was prepared in `01_data_prep.ipynb`.

In [ ]:
job = client.fine_tuning.jobs.create(
    model="ministral-3b-latest",
    job_type="classifier",
    training_files=[{"file_id": training_file_id, "weight": 1}],
    validation_files=[validation_file_id],
    hyperparameters={
        "training_steps": 100,
        "learning_rate": 0.00004,
    },
    auto_start=True,
)

console.print(Panel(
    f"[bold]Job ID:[/bold]  [cyan]{job.id}[/cyan]\n"
    f"[bold]Status:[/bold]  [yellow]{job.status}[/yellow]\n"
    f"[bold]Model:[/bold]   ministral-3b-latest\n"
    f"[bold]Steps:[/bold]   100\n"
    f"[dim]Typical training time: 5–15 minutes[/dim]",
    title="Fine-tuning job created",
    border_style="cyan",
))

## 4. Poll for completion

Checks job status every 30 seconds until the job reaches a terminal state.
Training typically completes in 5–15 minutes.

In [ ]:
POLL_INTERVAL   = 30   # seconds
TERMINAL_STATES = {"SUCCESS", "FAILED", "CANCELLED"}

console.print(f"Polling job [cyan]{job.id}[/cyan] every {POLL_INTERVAL}s …\n")

while True:
    status_obj = client.fine_tuning.jobs.get(job_id=job.id)
    status     = status_obj.status
    ts         = time.strftime("%H:%M:%S")
    console.print(f"  [{ts}]  status = [yellow]{status}[/yellow]")
    if status in TERMINAL_STATES:
        break
    time.sleep(POLL_INTERVAL)

if status != "SUCCESS":
    raise RuntimeError(
        f"Fine-tuning job ended with status: {status}\n"
        f"Check the Mistral dashboard for details: https://console.mistral.ai/finetuning"
    )

fine_tuned_model_id = status_obj.fine_tuned_model
console.print(f"\n[bold green]Training complete![/bold green]")
console.print(f"Fine-tuned model ID: [bold cyan]{fine_tuned_model_id}[/bold cyan]")

In [ ]:
# ── Training metrics from job events ─────────────────────────────────────
events = client.fine_tuning.jobs.list_events(job_id=job.id)
metric_events = [
    e for e in events.data
    if e.data and any(k in e.data for k in ("train_loss", "valid_loss", "loss"))
]

if metric_events:
    t = Table(title="Training metrics (final checkpoints)", header_style="bold")
    t.add_column("Step",       justify="right")
    t.add_column("Train loss", justify="right", style="green")
    t.add_column("Val loss",   justify="right", style="yellow")
    for e in metric_events[-5:]:
        d = e.data
        t.add_row(
            str(d.get("step", "—")),
            f"{d.get('train_loss', 0):.4f}",
            f"{d.get('valid_loss', d.get('loss', 0)):.4f}",
        )
    console.print(t)
else:
    console.print("[dim]No loss events in job history (normal for short runs).[/dim]")

## 5. Evaluate on test set

`test.jsonl` was held out in `01_data_prep.ipynb` and never uploaded for training.
We classify every test example in a single batched call, then report
per-class precision, recall, and F1 using `sklearn`.

> The test set contains ~9 examples (15 % of 60). F1 scores on a set this small
> have high variance — treat them as a directional signal, not a production benchmark.
> For a production deployment, collect ≥ 200 labelled examples and re-evaluate.

In [ ]:
# ── Load test set ─────────────────────────────────────────────────────────
test_examples: list[dict] = []
with open(TEST_FILE) as f:
    for line in f:
        test_examples.append(json.loads(line.strip()))

console.print(f"Loaded [bold]{len(test_examples)}[/bold] test examples")

# ── Batch classify ────────────────────────────────────────────────────────
response = client.classifiers.classify(
    model=fine_tuned_model_id,
    inputs=[{"text": ex["text"]} for ex in test_examples],
)

y_true: list[str] = []
y_pred: list[str] = []
rows: list[dict]  = []

for ex, result in zip(test_examples, response.results):
    true_label = ex["labels"]["intent"]
    probs      = result.probabilities            # dict[str, float]
    pred_label = max(probs, key=probs.get)
    confidence = probs[pred_label]

    y_true.append(true_label)
    y_pred.append(pred_label)
    rows.append({
        "true":       true_label,
        "predicted":  pred_label,
        "confidence": confidence,
        "correct":    true_label == pred_label,
        "text":       ex["text"][:80] + "…",
    })

# ── Per-example table ─────────────────────────────────────────────────────
t = Table(title="Test set predictions", header_style="bold", show_lines=False)
t.add_column("True",      style="cyan",   min_width=20)
t.add_column("Predicted", style="magenta",min_width=20)
t.add_column("Conf",      justify="right")
t.add_column("OK?",       justify="center")
t.add_column("Text",      style="dim",    max_width=55)

for row in rows:
    t.add_row(
        row["true"],
        row["predicted"],
        f"{row['confidence']:.0%}",
        "[green]✓[/green]" if row["correct"] else "[red]✗[/red]",
        row["text"],
    )

console.print(t)
accuracy = sum(r["correct"] for r in rows) / len(rows)
console.print(f"\nOverall accuracy: [bold]{accuracy:.0%}[/bold] ({sum(r['correct'] for r in rows)}/{len(rows)})")

In [ ]:
# ── sklearn classification report (precision / recall / F1 per class) ─────
report = classification_report(
    y_true,
    y_pred,
    labels=INTENT_LABELS,
    zero_division=0,
)
console.print(Panel(report, title="Classification report", border_style="green"))

In [ ]:
# ── Confusion matrix as a pandas DataFrame ────────────────────────────────
cm = confusion_matrix(y_true, y_pred, labels=INTENT_LABELS)
cm_df = pd.DataFrame(cm, index=INTENT_LABELS, columns=INTENT_LABELS)
cm_df.index.name   = "true \\ pred"

# Style: highlight diagonal (correct predictions) in green
def highlight_diagonal(val):
    return "background-color: #d4edda; font-weight: bold" if val > 0 else ""

styled = (
    cm_df.style
    .applymap(highlight_diagonal, subset=pd.IndexSlice[INTENT_LABELS, INTENT_LABELS])
    .set_caption("Confusion matrix — rows = true label, columns = predicted label")
)
display(styled)  # type: ignore  # noqa: F821  (display is a Jupyter built-in)

# Also print as plain text for non-Jupyter environments
console.print("\n[dim]Plain text confusion matrix:[/dim]")
console.print(cm_df.to_string())

## 6. Save model ID

In [ ]:
MODEL_ID_FILE.write_text(fine_tuned_model_id)
console.print(f"[bold green]Model ID saved →[/bold green] [cyan]{MODEL_ID_FILE}[/cyan]")
console.print(f"\n[bold]{fine_tuned_model_id}[/bold]")
console.print("\n[dim]This ID is loaded automatically by 03_agent_demo.ipynb and app.py.[/dim]")

# Update the config file so downstream notebooks have the full picture
if CONFIG_FILE.exists():
    cfg = json.loads(CONFIG_FILE.read_text())
    cfg["fine_tuned_model_id"] = fine_tuned_model_id
    CONFIG_FILE.write_text(json.dumps(cfg, indent=2))
    console.print(f"[dim]Also updated {CONFIG_FILE} with fine_tuned_model_id.[/dim]")

## 7. Key insight

This classifier replaces the intent-routing work that `mistral-large` would otherwise
have to do during the agentic loop. A dedicated 3B classifier is **faster, cheaper,
and more accurate on your specific taxonomy** than prompting a general-purpose model
to guess the intent — because it has been trained discriminatively on your exact
label set rather than relying on a large model's in-context pattern matching.

**Forge takes this further:** pre-training on your full document corpus — ticket
history, runbooks, architecture docs, internal wikis — means the generative model
itself develops *institutional fluency*, not just routing accuracy. Terms like
"Nexus", "Prism", "Helix", "prod-payments", and "SEV1" are no longer out-of-vocabulary
tokens that the model has to interpret from context; they are part of its learned
representation of the world. The Classifier Factory fine-tune we ran here is the
supervised, label-aware variant of the same principle — domain adaptation at the
classification layer rather than the full model weights.